# P7 Parts Demand — Training Notebook

**Goal:** Build a `failure_part_map` from historical intervention data  
`ordres_intervention.actual_failure_type` + `parts_replaced` text  
→ save `ml_model_p7_parts_demand.pkl` to both model dirs.

**Graduate gate:** P7 precision/recall/F1 must beat a naive global-popularity baseline.

In [1]:
import os, sys, re, warnings
from collections import defaultdict, Counter
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import shutil

warnings.filterwarnings('ignore')

ROOT = os.path.abspath('../../..')   # ml_research/ -> ml-microservice/ -> app/ -> repo root
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

MODELS_DIR    = os.path.join(ROOT, 'app', 'backend', 'modules', 'ml', 'models')
ML_SVC_MODELS = os.path.join(ROOT, 'app', 'ml-microservice', 'models')
DB_DIR        = os.path.join(ROOT, 'db_ml_training')
PKL_NAME      = 'ml_model_p7_parts_demand.pkl'

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(ML_SVC_MODELS, exist_ok=True)

print(f'ROOT          : {ROOT}')
print(f'DB_DIR        : {DB_DIR}  exists={os.path.exists(DB_DIR)}')
print(f'MODELS_DIR    : {MODELS_DIR}  exists={os.path.exists(MODELS_DIR)}')
print(f'ML_SVC_MODELS : {ML_SVC_MODELS}  exists={os.path.exists(ML_SVC_MODELS)}')

ROOT          : /workspace
DB_DIR        : /workspace/db_ml_training  exists=True
MODELS_DIR    : /workspace/app/backend/modules/ml/models  exists=True
ML_SVC_MODELS : /workspace/app/ml-microservice/models  exists=True


## Cell 1 — Load data

In [2]:
# --- primary source ---
oi = pd.read_csv(os.path.join(DB_DIR, 'ordres_intervention.csv'),
                 low_memory=False, encoding='latin-1')
print(f'ordres_intervention : {oi.shape}')
print('failure type counts :', oi['actual_failure_type'].value_counts().to_dict())
print('rows with parts_replaced :', oi['parts_replaced'].notna().sum())

# --- optional sources (may be empty) ---
def _load_or_empty(filename, cols):
    path = os.path.join(DB_DIR, filename)
    df = pd.read_csv(path, encoding='latin-1')
    if df.empty:
        print(f'{filename}: EMPTY')
    else:
        print(f'{filename}: {len(df)} rows')
    return df

pieces  = _load_or_empty('pieces.csv',         ['id','reference','name','min_stock'])
stock   = _load_or_empty('stock.csv',           ['piece_id','quantity'])
ms      = _load_or_empty('mouvement_stock.csv', ['piece_id','quantity','movement_type'])

ordres_intervention : (4000, 47)
failure type counts : {'TWF': 1000, 'HDF': 1000, 'PWF': 1000, 'OSF': 1000}
rows with parts_replaced : 4000
pieces.csv: EMPTY
stock.csv: EMPTY
mouvement_stock.csv: EMPTY


## Cell 2 — Parse `parts_replaced` → build parts catalog

In [ ]:
REF_PAT = re.compile(r'\(r[eé]f\.\s*([A-Z0-9\-]+)\)', re.IGNORECASE)

# Strip size/model specs so variants collapse to one generic category
# Examples stripped: ⌀8mm  6305-ZZ  7.5kW  25A  SPB-2500  R410A  M12x1.75  x2
SPEC_PAT = re.compile(
    r'(?:'
    r'⌀[\d\.]+\w*'                       # diameter: ⌀8mm
    r'|\bx\d+\b'                          # quantity: x2, x3
    r'|\b\d+[\w\-\.×/²³°%]*'             # numeric tokens: 6305-ZZ, 7.5kW, 25A, M12x1.75
    r'|\b[A-Z]{2,6}[-/]?\d+[\w\-\.]*\b'  # model codes: SPB-2500, R410A, HSK-A63, ISO-40
    r')',
    re.IGNORECASE
)

CONSUMABLE_KW = [
    'liquide', 'huile', 'graisse', 'filtre', 'joint', 'vis', 'bague',
    'gaz', 'colle', 'nettoyant', 'lubrifiant', 'pad thermique',
    'applicateur', 'isopropylique'
]

def normalize_name(raw: str) -> str:
    """Strip refs + size/model specs → generic category name."""
    s = REF_PAT.sub('', raw)          # remove (réf. XXX-NNN)
    s = re.sub(r'\([^)]*\)', '', s)   # remove any remaining parens
    s = SPEC_PAT.sub(' ', s)          # strip specs / model codes
    s = re.sub(r'\s+', ' ', s).strip().rstrip(',').strip()
    return s.lower()                   # lowercase for consistent keying

def is_consumable(name: str) -> bool:
    return any(kw in name for kw in CONSUMABLE_KW)

part_key_to_id: dict = {}
part_catalog: dict   = {}
_next_id = 1

def get_or_create_part(raw_part: str):
    global _next_id
    ref_m = REF_PAT.search(raw_part)
    ref   = ref_m.group(1) if ref_m else None
    name  = normalize_name(raw_part)
    if not name or name in ('', ',', '.'):
        return None
    key = name   # always key by normalized name (groups all size variants)
    if key not in part_key_to_id:
        pid = _next_id
        part_key_to_id[key] = pid
        part_catalog[pid] = {
            'piece_id'      : pid,
            'reference'     : ref or f'SYN-{pid:04d}',
            'name'          : name,
            'is_consumable' : is_consumable(name),
            'on_hand'       : 0,
            'min_stock'     : 2,
        }
        _next_id += 1
    return part_key_to_id[key]

# Filter to rows with both failure type and parts
oi_clean = oi[
    oi['parts_replaced'].notna() &
    oi['actual_failure_type'].notna() &
    (oi['parts_replaced'].str.strip() != '')
].copy().reset_index(drop=True)

oi_clean['part_ids'] = oi_clean['parts_replaced'].apply(
    lambda pr: [
        pid for p in str(pr).split(',')
        if (pid := get_or_create_part(p.strip())) is not None
    ]
)

print(f'Clean interventions     : {len(oi_clean)}')
print(f'Unique parts in catalog : {len(part_catalog)}')
print(f'  of which consumable   : {sum(1 for p in part_catalog.values() if p["is_consumable"])}')

# Sanity check: show top-5 most frequent parts per failure type
print()
from collections import Counter as _C
for ft in sorted(oi_clean['actual_failure_type'].unique()):
    sub = oi_clean[oi_clean['actual_failure_type'] == ft]
    cnt = _C(pid for pids in sub['part_ids'] for pid in pids)
    n   = len(sub)
    top = [(part_catalog[pid]['name'], round(c/n, 2)) for pid, c in cnt.most_common(3)]
    print(f'  {ft} top-3: {top}')

## Cell 3 — Build `failure_part_map`

For each failure type + part:  
- `p_used` = fraction of interventions of that type that included this part  
- `expected_qty` = 1.0 (text source has no quantity info)

In [ ]:
THETA = 0.05  # 5% — lower threshold needed for varied industrial part names

ft_total = Counter(oi_clean['actual_failure_type'])
ft_part_counts: dict = defaultdict(lambda: defaultdict(int))

for _, row in oi_clean.iterrows():
    ft = row['actual_failure_type']
    for pid in row['part_ids']:
        ft_part_counts[ft][pid] += 1

failure_part_map: dict = {}
for ft, part_counts in ft_part_counts.items():
    total = ft_total[ft]
    failure_part_map[ft] = {
        pid: {
            'p_used'         : round(count / total, 4),
            'expected_qty'   : 1.0,
            'piece_reference': part_catalog[pid]['reference'],
            'piece_name'     : part_catalog[pid]['name'],
        }
        for pid, count in part_counts.items()
        if count / total >= THETA
    }

print('failure_part_map summary:')
for ft, parts in sorted(failure_part_map.items()):
    top3 = sorted(parts.items(), key=lambda x: -x[1]['p_used'])[:3]
    print(f'  {ft}: {len(parts)} parts  |  top3: {[(v["piece_name"][:35], v["p_used"]) for _, v in top3]}')

## Cell 4 — Consumable params (Croston)

Computed from `mouvement_stock` out-movements.  
Empty here since mouvement_stock has no rows — consumable parts fall back to failure_part_map at inference.

In [5]:
consumable_params: dict = {}

if len(ms) > 0 and 'movement_type' in ms.columns:
    ms_out = ms[ms['movement_type'].str.lower() == 'out'].copy()
    if 'created_at' in ms_out.columns:
        ms_out['month'] = pd.to_datetime(ms_out['created_at'], errors='coerce').dt.to_period('M')
        for pid, grp in ms_out.groupby('piece_id'):
            monthly = grp.groupby('month')['quantity'].sum()
            series = monthly.reindex(
                pd.period_range(monthly.index.min(), monthly.index.max(), freq='M'), fill_value=0
            ).tolist()
            if len(series) >= 3:
                consumable_params[int(pid)] = {'series': series}
    print(f'consumable_params built from mouvement_stock: {len(consumable_params)} entries')
else:
    print('mouvement_stock empty — consumable_params = {} (Croston skipped)')
    print('Consumable parts will be served by failure_part_map at inference.')

print(f'consumable_params entries: {len(consumable_params)}')

mouvement_stock empty — consumable_params = {} (Croston skipped)
Consumable parts will be served by failure_part_map at inference.
consumable_params entries: 0


## Cell 5 — Backtest (graduate gate)

**Hold out last 25%** of interventions.  
Given `actual_failure_type`, predict parts using `failure_part_map`.  
Compare to naive baseline (always predict top-5 globally popular parts).  
**Graduate if P7 F1 > baseline F1.**

In [6]:
n = len(oi_clean)
split = int(n * 0.75)
holdout = oi_clean.iloc[split:].copy()
train   = oi_clean.iloc[:split].copy()
print(f'Train: {len(train)}  |  Holdout: {len(holdout)}')

def predict_parts(failure_type: str, theta: float = THETA) -> set:
    """Return piece_ids predicted for this failure type at given threshold."""
    return {
        pid for pid, stats in failure_part_map.get(failure_type, {}).items()
        if stats['p_used'] >= theta
    }

# P7 metrics
p7_prec, p7_rec = [], []
for _, row in holdout.iterrows():
    actual = set(row['part_ids'])
    if not actual:
        continue
    predicted = predict_parts(row['actual_failure_type'])
    if not predicted:
        p7_prec.append(0.0); p7_rec.append(0.0)
        continue
    tp = len(actual & predicted)
    p7_prec.append(tp / len(predicted))
    p7_rec.append(tp / len(actual))

P7_P = float(np.mean(p7_prec)) if p7_prec else 0.0
P7_R = float(np.mean(p7_rec))  if p7_rec  else 0.0
P7_F = 2 * P7_P * P7_R / (P7_P + P7_R + 1e-9)

# Naive baseline: always predict top-5 globally popular parts
all_part_counts = Counter(pid for pids in oi_clean['part_ids'] for pid in pids)
baseline_parts  = set(pid for pid, _ in all_part_counts.most_common(5))

b_prec, b_rec = [], []
for _, row in holdout.iterrows():
    actual = set(row['part_ids'])
    if not actual:
        continue
    tp = len(actual & baseline_parts)
    b_prec.append(tp / len(baseline_parts))
    b_rec.append(tp / len(actual))

B_P = float(np.mean(b_prec)) if b_prec else 0.0
B_R = float(np.mean(b_rec))  if b_rec  else 0.0
B_F = 2 * B_P * B_R / (B_P + B_R + 1e-9)

print(f'P7 Model  →  Precision: {P7_P:.3f}  Recall: {P7_R:.3f}  F1: {P7_F:.3f}')
print(f'Baseline  →  Precision: {B_P:.3f}  Recall: {B_R:.3f}  F1: {B_F:.3f}')

should_graduate = P7_F > B_F
print()
print(f'GRADUATE: {"YES ✓" if should_graduate else "NO ✗  (adjust THETA or review map)"}')

Train: 3000  |  Holdout: 1000
P7 Model  →  Precision: 0.000  Recall: 0.000  F1: 0.000
Baseline  →  Precision: 0.024  Recall: 0.030  F1: 0.027

GRADUATE: NO ✗  (adjust THETA or review map)


## Cell 6 — Save pkl (both model dirs)

In [7]:
if not should_graduate:
    raise RuntimeError('Graduate gate FAILED — fix metrics before saving pkl.')

payload = {
    'failure_part_map' : failure_part_map,
    'consumable_params': consumable_params,
    'parts_catalog'    : part_catalog,
    'meta': {
        'trained_at'    : datetime.utcnow().isoformat(),
        'n_interventions': len(oi_clean),
        'failure_types' : list(failure_part_map.keys()),
        'n_parts'       : len(part_catalog),
        'theta'         : THETA,
        'horizon_days'  : 30,
        'precision'     : round(P7_P, 4),
        'recall'        : round(P7_R, 4),
        'f1'            : round(P7_F, 4),
        'source'        : 'ordres_intervention.parts_replaced',
    }
}

# Save to backend models dir
pkl_path = os.path.join(MODELS_DIR, PKL_NAME)
joblib.dump(payload, pkl_path)
print(f'Saved  → {pkl_path}')

# Sync to ml-microservice models dir
svc_path = os.path.join(ML_SVC_MODELS, PKL_NAME)
shutil.copy2(pkl_path, svc_path)
print(f'Copied → {svc_path}')

# Verify
loaded = joblib.load(pkl_path)
assert 'failure_part_map' in loaded
assert 'consumable_params' in loaded
assert 'parts_catalog' in loaded
assert 'meta' in loaded
print()
print('✓ pkl verified — keys: failure_part_map, consumable_params, parts_catalog, meta')
print(f'  failure types    : {loaded["meta"]["failure_types"]}')
print(f'  parts in catalog : {loaded["meta"]["n_parts"]}')
print(f'  F1 score         : {loaded["meta"]["f1"]}')
print(f'  trained_at       : {loaded["meta"]["trained_at"]}')

RuntimeError: Graduate gate FAILED — fix metrics before saving pkl.